### Auditoria Gran Encuesta Integrada De Hogares 2026

**4 MODULOS **
- CGSE
- FT
- OCUPADOS
- DESEMPLEADOS


In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

In [0]:
# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()

In [0]:
cgse

In [0]:
#Leyendo la tabla de fuerza de trabajo
ft = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
ft=ft.toPandas()
ft

Exploracion demodulos y calculo base

### funciones basicas de pandas 

In [0]:
# Filtrando a nivel de columna
cgse['PERIODO']

#Aplicando operaciones
cgse['PERIODO'].sum()
cgse['PERIODO'].max()
cgse['PERIODO'].min()
cgse['PERIODO'].mean()
cgse['PERIODO'].std()
cgse['PERIODO'].var()
cgse['PERIODO'].median()
cgse['PERIODO'].mode()

In [0]:
cgse.head(20)

In [0]:
cgse.describe()

Filtrando varias columnas de la tabla nivel de columna

In [0]:
cgse[['PERIODO','FEX_C18','P6090']]

Filtro a nivel de fila, filtrar solo personas donde la variable DPTO=63

#Operadores de comparacion
"""
== : igual a
< :  menor a 
<= : menor o igual a 
 : mayor a 
= : mayor o igual a 
!= : diferente a

"""

In [0]:
cgse[cgse['DPTO']==63]
cgse_quindio = cgse[cgse['DPTO']==63]

In [0]:
cgse[(cgse['DPTO']==63) & (cgse['P3271']==63)]

In [0]:
cgse['P3271'].unique

In [0]:
cgse['DPTO'].unique()

## 1. poblacion total


In [0]:
cgse
PT = cgse['FEX_C18'].sum()
PT

Para junio de 2026, la tasa de desocupación del total nacional fue 8,0%


**Como verificamos esto ?**

La tasa de desempleo se calcula mediante la formula:

-> TD=D/PEA*100

- D : Cantidad de personas desempleadas
- PEA : poblacion economicamente activa

**Primer tema a resolver como calcular la cantidad de personas desempleadas ?**

In [0]:
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
fuerzadetrabajo = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")


In [0]:
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
fuerzadetrabajo=fuerzadetrabajo.toPandas()
ocupados=ocupados.toPandas()

clase de hoy 28 sept

In [0]:
import pandas

desempleados = pandas.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
MDQ=desempleados[(desempleados['P3271']==2) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()
HDQ=desempleados[(desempleados['P3271']==1) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()

print(f"en el quindio hay {MDQ} Mujeres desempleadas")
print(f"en el quindio hay {HDQ} Hombres desempleados")
     

In [0]:
## Calcular PEA de mujeres y hombres en el quindio
 
# 1.PEA de mujeres en el quindio
PEAMQ = fuerzadetrabajo[(fuerzadetrabajo['fuerzadetrabajo']==1) & (fuerzadetrabajo['DPTO']==63) & (fuerzadetrabajo['P3271']==2)]['FEX_C18'].sum()
# 2.PEA de hombres en el quindio
PEAHQ = fuerzadetrabajo[(fuerzadetrabajo['fuerzadetrabajo']==1) & (ft['DPTO']==63) & (fuerzadetrabajo['P3271']==1) ]['FEX_C18'].sum()
 
print(f"en el quindio hay {PEAMQ} Mujeres en la PEA")
print(f"en el quindio hay {PEAHQ} Hombres en la PEA")


In [0]:
fuerzadetrabajo= pandas.merge(fuerzadetrabajo, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
#Por ultimo cuando ya tenemos cantidad de desempleados y PEA podemos calcular el porcentaje de desempleo
 
TDHQ = HDQ/PEAHQ *100
print(f"El porcentaje de desempleo masculino en el quindio es de {TDHQ}%")
TDMQ = MDQ/PEAMQ *100
print(f"El porcentaje de desempleo femenino en el quindio es de {TDMQ}%")